In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import os,glob
import datetime

In [2]:
ix = pd.read_csv('https://data-argo.ifremer.fr/ar_index_global_prof.txt', 
                 sep=',', index_col=None, header=0, skiprows=8,
                 names=['file','date','latitude','longitude','ocean','profiler_type','institution','update'], 
                 dtype={'file': np.unicode_, 'latitude': np.float32, 'longitude': np.float32, 'ocean': str, 'profiler_type': str, 'institution': str}
                )
ix.tail()

,file,date,latitude,longitude,ocean,profiler_type,institution,update
3104494,nmdis/2901633/profiles/R2901633_067.nc,2.013050e+13,27.462000,139.106995,P,841,NM,20130507103443
3104495,nmdis/2901633/profiles/R2901633_068.nc,2.013051e+13,27.431999,138.839996,P,841,NM,20130511165723
3104496,nmdis/2901633/profiles/R2901633_069.nc,2.013052e+13,27.691999,138.677002,P,841,NM,20130521170139
3104497,nmdis/2901633/profiles/R2901633_070.nc,2.013053e+13,27.895000,138.464996,P,841,NM,20130531181516
3104498,nmdis/2901633/profiles/R2901633_071.nc,2.013061e+13,27.931000,138.089996,P,841,NM,20130617181801


In [3]:
ixs = ix[(ix['profiler_type']=='838')]

ixs = ixs.reset_index().drop(columns='index')
# parse the date after the subset, much more fast than inside the pd.read_csv()
ixs['date']= pd.to_datetime(ixs['date'],format='%Y%m%d%H%M%S')
# period of interest [datemin, datemax]
poi = np.array(['2000-01-01','2023-11-30'],dtype='datetime64')
ixs = ixs[(ixs['date']>=poi[0])&(ixs['date']<=poi[1])].reset_index()

# dac generation
dacs = {'AO':'aoml','BO':'bodc','IF':'coriolis','HZ':'csio','CS':'csiro','IN':'incois','JA':'jma','KM':'kma','KO':'kordi','ME':'meds','NM':'nmdis'}
ixs['wmo']=[int(f.split('/')[1]) for f in ixs['file']]
ixs['dac']=[dacs[f] for f in ixs['institution']]
ixs = ixs.groupby('wmo').last().reset_index()
ixs.tail()

,wmo,index,file,date,latitude,longitude,ocean,profiler_type,institution,update,dac
125,6990538,10914,coriolis/6990538/profiles/R6990538_016.nc,2023-11-22 00:06:00,53.640999,-26.841999,A,838,IF,20231122012547,coriolis
126,6990627,10978,coriolis/6990627/profiles/R6990627_019.nc,2022-08-15 06:05:00,29.014999,-16.138000,A,838,IF,20240904130240,coriolis
127,6990628,11048,coriolis/6990628/profiles/R6990628_069.nc,2023-07-26 06:17:00,29.329000,-15.442000,A,838,IF,20240904130708,coriolis
128,7901036,11135,coriolis/7901036/profiles/R7901036_015.nc,2023-11-22 16:09:00,59.508999,-48.191002,A,838,IF,20231122172659,coriolis
129,7901037,11195,coriolis/7901037/profiles/R7901037_015.nc,2023-11-21 18:15:00,58.676998,-41.412998,A,838,IF,20231121192657,coriolis


In [ ]:
for i in range(len(ixs)) :   
    
    